In [ ]:
#==== Compute ECI===

#!/usr/bin/env python3
"""
The following code is a modified version of the open-source code provided by Epoch AI, adapted for local use.
Source:https://github.com/epoch-research/eci-public

Fit ECI model and save scores to outputs directory.

Usage:
    python scripts/fit_eci.py
    python scripts/fit_eci.py --input path/to/benchmarks.csv
    python scripts/fit_eci.py --bootstrap-samples 200
    python scripts/fit_eci.py --numeric-jacobian
"""

#import argparse
#from pathlib import Path
#from eci import load_benchmark_data, fit_eci_model, compute_eci_scores

import sys, os
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from pathlib import Path
import argparse

# Folder configulation
sys.path.append(os.getcwd()) 
from eci.dataloader import prepare_benchmark_data as load_benchmark_data
from eci.fitting import fit_eci_model, compute_eci_scores


#DEFAULT_INPUT_URL = "https://epoch.ai/data/eci_benchmarks.csv"
DEFAULT_INPUT_URL = "eci_benchmarks.csv"
#OUTPUT_DIR = Path(__file__).parent.parent / "outputs"
OUTPUT_DIR = Path("outputs")

def main():
    parser = argparse.ArgumentParser(description="Fit ECI model and save scores")
    parser.add_argument(
        "--input",
        default=DEFAULT_INPUT_URL,
        help=f"Path or URL to benchmark data CSV (default: {DEFAULT_INPUT_URL})",
    )
    parser.add_argument(
        "--bootstrap-samples",
        type=int,
        default=100,
        help="Number of bootstrap samples for confidence intervals (default: 100)",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=OUTPUT_DIR,
        help=f"Output directory for scores (default: {OUTPUT_DIR})",
    )
    parser.add_argument(
        "--numeric-jacobian",
        action="store_true",
        help="Use numerical Jacobian instead of analytical (slower, matches website exactly)",
    )
    args = parser.parse_args(args=[]) 
    
    print(f"Loading benchmark data from {args.input}...")
# If the input is a local file, read it directly; otherwise, use the function
    if os.path.exists(args.input):
        df = pd.read_csv(args.input)
    else:
        df = load_benchmark_data(args.input)
        print(f"  Downloaded from URL: {args.input}")
    
    print(f"  Loaded {len(df)} performance records")
    print(f"  {df['model_id'].nunique()} models, {df['benchmark_id'].nunique()} benchmarks")

    jacobian_type = "numerical" if args.numeric_jacobian else "analytical"
    print(f"\nFitting IRT model ({jacobian_type} Jacobian, {args.bootstrap_samples} bootstrap samples)...")
    model_df, bench_df = fit_eci_model(
        df,
        bootstrap_samples=args.bootstrap_samples,
        bootstrap_seed=12345,
        use_analytical_jacobian=not args.numeric_jacobian,
    )

    print("Computing ECI/EDI scores...")
    eci_df, edi_df = compute_eci_scores(model_df, bench_df)

    # Prepare output directory
    args.output_dir.mkdir(parents=True, exist_ok=True)

    # Save ECI scores
    eci_output = args.output_dir / "eci_scores.csv"
    eci_cols = ["Model", "eci", "eci_ci_low", "eci_ci_high"]
    eci_df[eci_cols].to_csv(eci_output, index=False)
    print(f"\nSaved ECI scores to {eci_output}")

    # Save EDI scores
    edi_output = args.output_dir / "edi_scores.csv"
    edi_cols = ["benchmark", "edi", "discriminability_scaled", "is_anchor"]
    if "benchmark_release_date" in edi_df.columns:
        edi_cols.insert(3, "benchmark_release_date")
    edi_df[edi_cols].to_csv(edi_output, index=False)
    print(f"Saved EDI scores to {edi_output}")

    
    # Print summary
    print("\n=== Top 10 Models by ECI ===")
    print(eci_df[["Model", "eci"]].head(10).to_string(index=False))
 
    return eci_df, model_df  

if __name__ == "__main__":
    # # Modify the code to execute `main()` and receive the return value
    eci_df, model_df = main() 

'''
The following code  processes AI benchmark data to estimate 
and visualize the annual growth rate of AI capabilities 
by fitting an exponential growth model to historical performance scores over time.
'''

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from scipy.optimize import curve_fit

# ==========================================
# CONFIGURATION & PARAMETERS
# ==========================================
# File paths
INPUT_BENCHMARK_CSV = "eci_benchmarks.csv"
OUTPUT_DIRECTORY = "outputs"
MERGED_DATA_FILENAME = "processed_capability_data.csv"

# Initial Guesses for Curve Fitting [Amplitude (a), Growth Rate (v), Offset]
# These are based on the scale of ECI scores (roughly 60 to 160)
INITIAL_GUESS = [100.0, 0.15, 0.0] 

# Fitting Bounds: ([min_a, min_v, min_offset], [max_a, max_v, max_offset])
# v is bounded between 0% and 200% annual growth
PARAMETER_BOUNDS = ([0.1, 0.0, -0.0001], [1000.0, 4.0, 0.0001]) 
# Optimization settings
MAX_FUNCTION_EVALUATIONS = 20000

# ==========================================
# DATA PREPARATION
# ==========================================
print("Step 1: Merging ECI scores with release dates...")

try:
    # Reload original benchmark data to recover 'date' information
    raw_benchmark_df = pd.read_csv(INPUT_BENCHMARK_CSV)
    
    # Get the latest release date for each model to handle duplicates
    model_release_dates = (
        raw_benchmark_df[['Model', 'date']]
        .drop_duplicates()
        .groupby('Model')
        .max()
        .reset_index()
    )

    # Combine the ECI scores (from variable 'eci_df') with their release dates using a Left Join
    capability_time_series_df = eci_df.merge(model_release_dates, on='Model', how='left')
    
    # Clean data: Convert to datetime and drop missing values
    capability_time_series_df['date'] = pd.to_datetime(capability_time_series_df['date'], errors='coerce')

    # Identify rows where either the date or the ECI score is missing (NaN)
    missing_mask = capability_time_series_df[['date', 'eci']].isna().any(axis=1)
    missing_rows = capability_time_series_df.loc[missing_mask, ['Model', 'date', 'eci']]

    print(f"Rows with missing in 'date' or 'eci': {len(missing_rows)}")
    display(missing_rows)   

    # Filter out the missing rows and sort the remaining data chronologically
    capability_time_series_df_clean = (
    capability_time_series_df.loc[~missing_mask].sort_values('date')
    )
    
    # Convert dates into a numerical format 't' (years elapsed since the very first model),using 365.25 to account for leap years.
    earliest_release_date = capability_time_series_df_clean['date'].min()
    capability_time_series_df_clean['years_since_start'] = (
    (capability_time_series_df_clean['date'] - earliest_release_date).dt.days / 365.25
    )
    
    # Ensure the output folder exists, then save the cleaned data to a CSV
    if not os.path.exists(OUTPUT_DIRECTORY):
        os.makedirs(OUTPUT_DIRECTORY)
    
    save_path = os.path.join(OUTPUT_DIRECTORY, MERGED_DATA_FILENAME)
    capability_time_series_df_clean.to_csv(save_path, index=False)
    print(f"✅ Merged data saved to: {save_path}")

# ==========================================
# EXPONENTIAL MODEL FITTING
# ==========================================
    print("\nStep 2: Fitting exponential growth model to industry average...")

    # Define the model function: Capability = Amplitude * (1 + annual_growth)^years + offset
    def exponential_growth_model(t, amplitude, annual_growth_rate, baseline_offset):
        return amplitude * (1 + annual_growth_rate)**t + baseline_offset

    # Perform Non-linear Least Squares (NLS) fitting to find the parameters (a, v, offset) that best fit the observed data points.
    optimized_params, covariance_matrix = curve_fit(
        exponential_growth_model,
        capability_time_series_df_clean['years_since_start'],
        capability_time_series_df_clean['eci'],
        p0=INITIAL_GUESS,
        bounds=PARAMETER_BOUNDS,
        maxfev=MAX_FUNCTION_EVALUATIONS
    )

    # Extract the resulting optimized values
    fitted_amplitude, fitted_growth_rate_v, fitted_offset = optimized_params

    # Calculate the Standard Error (uncertainty) of the parameters.
    # The square root of the diagonal of the covariance matrix gives the standard deviation.
    standard_errors = np.sqrt(np.diag(covariance_matrix))
    growth_rate_standard_deviation = standard_errors[1] # Index 1 corresponds to 'v'

    # Display results for Monte Carlo simulation
    print("-" * 60)
    print(f"ESTIMATED PARAMETERS FOR MONTE CARLO:")
    print(f"Mean Growth Rate (μ_v): {fitted_growth_rate_v:.4f} ({fitted_growth_rate_v*100:.2f}% per year)")
    print(f"Std Deviation (σ_v):   {growth_rate_standard_deviation:.4f}")
    print("-" * 60)
    print(f"Baseline Offset: {fitted_offset:.4f}")
    print(f"Amplitude (a):   {fitted_amplitude:.4f}")

# ==========================================
# VISUALIZATION
# ==========================================
    plt.figure(figsize=(14, 8))
    
    # Plot individual model data points
    plt.scatter(
        capability_time_series_df_clean['years_since_start'], 
        capability_time_series_df_clean['eci'], 
        color='gray', alpha=0.3, s=30, label='Individual AI Models'
    )
    
    # Calculate and plot yearly average for visual reference
    capability_time_series_df_clean['year_bin'] = capability_time_series_df_clean['years_since_start'].round(0)
    yearly_average_scores = capability_time_series_df_clean.groupby('year_bin')['eci'].mean()
    plt.scatter(
        yearly_average_scores.index, yearly_average_scores.values, 
        color='red', s=120, marker='X', label='Calculated Yearly Average', zorder=5
    )

    # Create a smooth line for the fitted exponential curve
    # Generate 100 points between 'time 0' and the latest date
    time_range = np.linspace(0, capability_time_series_df_clean['years_since_start'].max() + 0.5, 100)
    predicted_eci_scores = exponential_growth_model(time_range, *optimized_params)
    
    plt.plot(
        time_range, predicted_eci_scores, 
        color='blue', lw=3, label=f'Exponential Fit (v={fitted_growth_rate_v:.2%}/yr)'
    )


    # --- Convert relative years to calendar years ---
    # 1. Get the base year from the earliest data point (e.g., 2023)
    start_calendar_year = earliest_release_date.year
    max_t = capability_time_series_df_clean['years_since_start'].max()
    
    # 2. Define tick positions (0, 1, 2...) and their matching year labels (2023, 2024...)
    tick_positions = np.arange(0, int(np.ceil(max_t)) + 1, 1.0)
    tick_labels = [int(start_calendar_year + t) for t in tick_positions]

    plt.title('AI Capability Evolution: Exponential Trend Analysis ', fontsize=16)
    
    # 3. Apply the custom ticks and change the label
    plt.xticks(tick_positions, tick_labels)
    plt.xlabel('Year', fontsize=12)
    # ----------------------------------------------------------------
    
    plt.title('AI Capability Evolution: Exponential Trend Analysis (ECI)', fontsize=16)
    plt.ylabel('Capability Score (ECI Index)', fontsize=12)
    plt.legend(loc='upper left', frameon=True)
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plt.show()

except Exception as error:
    print(f"❌ An error occurred during processing: {error}")

Loading benchmark data from eci_benchmarks.csv...
  Loaded 1410 performance records
  160 models, 40 benchmarks

Fitting IRT model (analytical Jacobian, 100 bootstrap samples)...


Bootstrap:  72%|███████████████████████████████████████████████▌                  | 72/100 [01:07<00:22,  1.26sample/s]